In [ ]:
# COFEU SFT — Türkçe Sohbet + Bilgi (Colab)
# GPU: T4/A100 | eğitim ~30-45 dk, veri indirme ~5 dk
#
# Veri: AhiskaAI/sharegpt-turkish (gerçek sohbet, 74K tur)
#     + tascib/turkish-instruction (kısa soru/cevap, 324K)
# NOT: sft.py, veri hazırlığını python/prepare_sft_data.py ile yapıyor.

In [ ]:
# GPU zorunlu: CPU'da 29M model 116K ornek icin saatlerce surer.
# Kolayca "Torch not compiled with CUDA enabled" hatasi alinir (CPU-only
# torch kurulu gelebiliyor). O durumda CUDA'li wheel'i kendimiz kuruyoruz.
# Not: kernel'de zaten yuklenmis torch degismeyecek; egitim ayri surecte
# basladigi icin yeni kurulum orada gecerli olur, runtime yeniden
# baslatmaya gerek yok.
import subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
      or "GPU YOK -> Colab runtime ayarindan GPU sec")

def cuda_ok() -> bool:
    r = subprocess.run([sys.executable, "-c",
                        "import torch; print(torch.cuda.is_available())"],
                       capture_output=True, text=True)
    return r.stdout.strip() == "True"

print("torch CUDA:", cuda_ok())
if not cuda_ok():
    print("CUDA'li torch kuruluyor (cu121)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    "torch", "--index-url",
                    "https://download.pytorch.org/whl/cu121"], check=True)
    print("kuruldu, tekrar kontrol:", cuda_ok())

assert cuda_ok(), ("CUDA hala yok. Runtime ayarindan GPU/T4 secip "
                   "runtime'i yeniden baslat, sonra bu hucreyi tekrar calistir.")
print("GPU HAZIR")

In [ ]:
import subprocess
r = subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/cofeu/cofeuai.git", "/content/cofeuai"],
                   capture_output=True, text=True)
print(r.stderr[-300:])
if r.returncode != 0:
    raise RuntimeError("repo klonlanamadi (internet/DNS kontrolu)")
print(subprocess.run(["git", "-C", "/content/cofeuai", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
import base64, pathlib
# GOMULU KOPYA: python/sft.py -> /content/cofeuai/python/sft.py
# Dosya GitHub'a push edilmedi; notebook kendi kopyasini yaziyor.
# Ureten: data/sft/build_chat_notebook.py (python/sft.py okunarak)
_b64 = (
    "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJJbnN0cnVjdGlvbiB0dW5pbmcgKFNGVCkgLSBBbHBhY2EgYmnDp2ltbGkgdmVyaSBz"
    "ZXRpIGlsZS4KClRlbWVsIGXEn2l0aW1kZW4gKHRyYWluLnB5KSBmYXJrxLE6IHTDvG0gdG9rZW4nbGFyIGXFn2l0IGHEn8SxcmzE"
    "sWtsYSBkZcSfaWwsIFNBREVDRQphc2lzdGFuIGNldmFixLEga2F5xLFwIGhlc2FixLFuYSBnaXJlci4gUHJvbXB0IHRva2VuJ2xh"
    "csSxIC0xMDAgaWxlIG1hc2tlbGVuaXIKKFB5VG9yY2gnaW4gaWdub3JlX2luZGV4J2kpLCBiw7Z5bGVjZSBtb2RlbCBjZXZhYsSx"
    "IMO8cmV0bWV5aSDDtsSfcmVuaXIuCgpLdWxsYW7EsW06CiAgICBweXRob24gc2Z0LnB5IC0tZGF0YSBhbHBhY2EuanNvbmwgLS1y"
    "ZXN1bWUgLS1lcG9jaHMgMgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmlt"
    "cG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBtYXRoCmltcG9ydCByYW5kb20KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGli"
    "IGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBPcHRpb25hbCwgVHVwbGUKCmltcG9ydCB0b3JjaAoKZnJvbSBt"
    "b2RlbCBpbXBvcnQgQ29mZXVUcmFuc2Zvcm1lciwgTW9kZWxDb25maWcKZnJvbSB0b2tlbml6ZXIgaW1wb3J0IEJQRVRva2VuaXpl"
    "cgoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgJShsZXZlbG5hbWUp"
    "cyAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJjb2ZldS5zZnQiKQoKUk9PVCA9IFBhdGgoX19maWxl"
    "X18pLnJlc29sdmUoKS5wYXJlbnQucGFyZW50CklHTk9SRSA9IC0xMDAKClNZU1RFTV9QUk9NUFQgPSAiU2VuIHlhcmTEsW1jxLEg"
    "YmlyIHlhcGF5IHpla2EgYXNpc3RhbsSxc8Sxbi4iCgoKZGVmIGJ1aWxkX3Byb21wdChpbnN0cnVjdGlvbjogc3RyLCBpbnA6IHN0"
    "cikgLT4gc3RyOgogICAgIiIiS3VsbGFuxLFjxLEga8Sxc23EsW7EsSBrdXJhciAobW9kZWxpbiBnw7ZyZWNlxJ9pIGJhxJ9sYW0p"
    "LiIiIgogICAgaWYgaW5wIGFuZCBpbnAuc3RyaXAoKToKICAgICAgICByZXR1cm4gZiJ7aW5zdHJ1Y3Rpb259XG5cbntpbnAuc3Ry"
    "aXAoKX0iCiAgICByZXR1cm4gaW5zdHJ1Y3Rpb24uc3RyaXAoKQoKCmRlZiBidWlsZF90ZXh0KGluc3RydWN0aW9uOiBzdHIsIGlu"
    "cDogc3RyLCBvdXRwdXQ6IHN0cikgLT4gVHVwbGVbc3RyLCBzdHJdOgogICAgIiIiKHByb21wdCwgdGFtX21ldGluKSBkw7ZuZMO8"
    "csO8ci4KCiAgICB0YW1fbWV0aW4sIG1vZGVsaW4gw7bEn3Jlbm1lc2kgZ2VyZWtlbiB0YW0ga29udcWfbWEuIHByb21wdCwgY2V2"
    "YWLEsW4KICAgIGJhxZ9sYWTEscSfxLEgeWVyZSBrYWRhciBvbGFuIGvEsXPEsW0gdmUgbWFza2VsZW5pci4KCiAgICBOT1Q6IENl"
    "dmFwIHNvbnVuZGFraSA8fGVvc3w+LCBlbmNvZGUgc8SxcmFzxLFuZGEgbWV0aW4gb2xhcmFrIGRlxJ9pbAogICAgVEVLIFRPS0VO"
    "IG9sYXJhayBla2xlbm1lbGlkaXI7IGVuY29kZSgpIMO2emVsIHRva2VuJ2xhcsSxIHRhbsSxbWF6LAogICAgIjx8ZW9zfD4iIHlh"
    "esSxc8SxbsSxIHVuayArIHBhcsOnYWxhcmEgYsO2bGVyLgogICAgIiIiCiAgICBwID0gYnVpbGRfcHJvbXB0KGluc3RydWN0aW9u"
    "LCBpbnApCiAgICBwcm9tcHRfcGFydCA9ICgKICAgICAgICBmIiMjIyBTaXN0ZW06XG57U1lTVEVNX1BST01QVH1cblxuIgogICAg"
    "ICAgIGYiIyMjIEt1bGxhbsSxY8SxOlxue3B9XG5cbiIKICAgICAgICBmIiMjIyBBc2lzdGFuOlxuIgogICAgKQogICAgcmV0dXJu"
    "IHByb21wdF9wYXJ0LCBwcm9tcHRfcGFydCArIGYie291dHB1dC5zdHJpcCgpfSIKCgpkZWYgZW5jb2RlX2V4YW1wbGUodG9rLCBp"
    "bnN0cnVjdGlvbjogc3RyLCBpbnA6IHN0ciwgb3V0cHV0OiBzdHIsCiAgICAgICAgICAgICAgICAgICBibG9ja19zaXplOiBpbnQp"
    "IC0+IE9wdGlvbmFsW1R1cGxlW0xpc3RbaW50XSwgTGlzdFtpbnRdXV06CiAgICAiIiJCaXIgw7ZybmXEn2kgKGlkcywgbGFiZWxz"
    "KSDDp2V2aXJpci4gQmxvY2snYSBzxLHEn23EsXlvcnNhIE5vbmUuCgogICAgQ2V2YWLEsSBFT1MgaWxlIGthcGF0xLFyOiBtb2Rl"
    "bCBiw7Z5bGVjZSAiZHVyIiBpxZ9hcmV0aW5pIMO2xJ9yZW5pci4gSGFtIG1ldGluCiAgICBlxJ9pdGltaW5kZSBFT1MgaGnDpyBn"
    "w7Zyw7xsbWVkacSfaSBpw6dpbiBidSBrcml0aWsuCiAgICAiIiIKICAgIHByb21wdF9wYXJ0LCBmdWxsID0gYnVpbGRfdGV4dChp"
    "bnN0cnVjdGlvbiwgaW5wLCBvdXRwdXQpCiAgICBmdWxsX2lkcyA9IHRvay5lbmNvZGUoZnVsbCkKICAgIGlmIHRvay5lb3NfaWQg"
    "Pj0gMDoKICAgICAgICBmdWxsX2lkcyA9IGZ1bGxfaWRzICsgW3Rvay5lb3NfaWRdCiAgICBpZiBsZW4oZnVsbF9pZHMpID4gYmxv"
    "Y2tfc2l6ZToKICAgICAgICByZXR1cm4gTm9uZQogICAgcHJvbXB0X2xlbiA9IGxlbih0b2suZW5jb2RlKHByb21wdF9wYXJ0KSkK"
    "ICAgIGxhYmVscyA9IGxpc3QoZnVsbF9pZHMpCiAgICBmb3IgaSBpbiByYW5nZShtaW4ocHJvbXB0X2xlbiwgbGVuKGxhYmVscykp"
    "KToKICAgICAgICBsYWJlbHNbaV0gPSBJR05PUkUKICAgIHJldHVybiBmdWxsX2lkcywgbGFiZWxzCgoKZGVmIGxvYWRfanNvbmwo"
    "cGF0aDogUGF0aCkgLT4gTGlzdFtkaWN0XToKICAgIHJvd3MgPSBbXQogICAgd2l0aCBvcGVuKHBhdGgsICJyIiwgZW5jb2Rpbmc9"
    "InV0Zi04IikgYXMgZjoKICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAg"
    "ICAgICAgIGlmIG5vdCBsaW5lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgb2JqID0ganNvbi5sb2Fkcyhs"
    "aW5lKQogICAgICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAiaW5zdHJ1Y3Rpb24iOiBvYmouZ2V0KCJpbnN0"
    "cnVjdGlvbiIsICIiKSwKICAgICAgICAgICAgICAgICJpbnB1dCI6IG9iai5nZXQoImlucHV0IiwgIiIpLAogICAgICAgICAgICAg"
    "ICAgIm91dHB1dCI6IG9iai5nZXQoIm91dHB1dCIsICIiKSwKICAgICAgICAgICAgfSkKICAgIHJldHVybiByb3dzCgoKZGVmIG1h"
    "a2VfYmF0Y2hlcyhzYW1wbGVzLCBiYXRjaF9zaXplOiBpbnQsIGJsb2NrX3NpemU6IGludCwgcm5nOiByYW5kb20uUmFuZG9tKToK"
    "ICAgICIiInNhbXBsZXM6IChpZHMsIGxhYmVscykgbGlzdGVzaSAtPiAoeCwgeSkgdGVuc29yIGJhdGNoJ2xlcmkuIiIiCiAgICBp"
    "ZHhzID0gbGlzdChyYW5nZShsZW4oc2FtcGxlcykpKQogICAgcm5nLnNodWZmbGUoaWR4cykKICAgIGZvciBiIGluIHJhbmdlKDAs"
    "IGxlbihpZHhzKSwgYmF0Y2hfc2l6ZSk6CiAgICAgICAgY2h1bmsgPSBbc2FtcGxlc1tpXSBmb3IgaSBpbiBpZHhzW2I6YiArIGJh"
    "dGNoX3NpemVdXQogICAgICAgIHBhZCA9IHRva19wYWRfaWRfaG9sZGVyWzBdCiAgICAgICAgbWF4bGVuID0gbWF4KGxlbihpZHMp"
    "IGZvciBpZHMsIF8gaW4gY2h1bmspCiAgICAgICAgbWF4bGVuID0gbWluKG1heGxlbiwgYmxvY2tfc2l6ZSkKICAgICAgICB4ID0g"
    "dG9yY2guZnVsbCgobGVuKGNodW5rKSwgbWF4bGVuKSwgcGFkLCBkdHlwZT10b3JjaC5sb25nKQogICAgICAgIHkgPSB0b3JjaC5m"
    "dWxsKChsZW4oY2h1bmspLCBtYXhsZW4pLCBJR05PUkUsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICAgICAgZm9yIGosIChpZHMsIGxh"
    "YmVscykgaW4gZW51bWVyYXRlKGNodW5rKToKICAgICAgICAgICAgbiA9IG1pbihsZW4oaWRzKSwgbWF4bGVuKQogICAgICAgICAg"
    "ICB4W2osIDpuXSA9IHRvcmNoLnRlbnNvcihpZHNbOm5dLCBkdHlwZT10b3JjaC5sb25nKQogICAgICAgICAgICB5W2osIDpuXSA9"
    "IHRvcmNoLnRlbnNvcihsYWJlbHNbOm5dLCBkdHlwZT10b3JjaC5sb25nKQogICAgICAgIHlpZWxkIHgsIHkKCgp0b2tfcGFkX2lk"
    "X2hvbGRlciA9IFsyXQoKCmRlZiBldmFsdWF0ZShtb2RlbCwgc2FtcGxlcywgYmF0Y2hfc2l6ZTogaW50LCBibG9ja19zaXplOiBp"
    "bnQsIGRldmljZTogc3RyLAogICAgICAgICAgICAgbWF4X2JhdGNoZXM6IGludCA9IDIwKSAtPiBmbG9hdDoKICAgIG1vZGVsLmV2"
    "YWwoKQogICAgcm5nID0gcmFuZG9tLlJhbmRvbSgxMjM0KQogICAgdG90YWxfbG9zcywgdG90YWxfbiA9IDAuMCwgMAogICAgd2l0"
    "aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIGJpLCAoeCwgeSkgaW4gZW51bWVyYXRlKG1ha2VfYmF0Y2hlcyhzYW1wbGVz"
    "LCBiYXRjaF9zaXplLCBibG9ja19zaXplLCBybmcpKToKICAgICAgICAgICAgaWYgYmkgPj0gbWF4X2JhdGNoZXM6CiAgICAgICAg"
    "ICAgICAgICBicmVhawogICAgICAgICAgICB4LCB5ID0geC50byhkZXZpY2UpLCB5LnRvKGRldmljZSkKICAgICAgICAgICAgXywg"
    "bG9zcyA9IG1vZGVsKHgsIHRhcmdldHM9eSkKICAgICAgICAgICAgbiA9ICh5ICE9IElHTk9SRSkuc3VtKCkuaXRlbSgpCiAgICAg"
    "ICAgICAgIGlmIG4gPiAwOgogICAgICAgICAgICAgICAgdG90YWxfbG9zcyArPSBsb3NzLml0ZW0oKSAqIG4KICAgICAgICAgICAg"
    "ICAgIHRvdGFsX24gKz0gbgogICAgbW9kZWwudHJhaW4oKQogICAgcmV0dXJuIHRvdGFsX2xvc3MgLyBtYXgodG90YWxfbiwgMSkK"
    "CgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iSW5zdHJ1Y3Rpb24gdHVu"
    "aW5nIChTRlQpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhIiwgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUsIGhlbHA9IkpT"
    "T05MIHZlcmkgZG9zeWFzxLEiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dC1kaXIiLCB0eXBlPXN0ciwgZGVmYXVsdD1zdHIo"
    "Uk9PVCAvICJjaGVja3BvaW50cyIgLyAic2Z0IikpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVzdW1lIiwgdHlwZT1zdHIsIGRl"
    "ZmF1bHQ9c3RyKFJPT1QgLyAiY2hlY2twb2ludHMiIC8gImNvZmV1LnB0IiksCiAgICAgICAgICAgICAgICAgICAgaGVscD0iQmHF"
    "n2xhbmfEscOnIGNoZWNrcG9pbnQnaSAodGVtZWwgZcSfaXRpbSDDp8Sxa3TEsXPEsSkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCIt"
    "LWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50"
    "LCBkZWZhdWx0PTgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTJlLTQsIGhlbHA9IlNG"
    "VCBpw6dpbiBkYWhhIGTDvMWfw7xrIGxyIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ncmFkLWNsaXAiLCB0eXBlPWZsb2F0LCBk"
    "ZWZhdWx0PTEuMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXgtaXRlcnMiLCB0eXBlPWludCwgZGVmYXVsdD0wLCBoZWxwPSIw"
    "ID0gZXBvY2ggc2F5xLFzxLEga2FkYXIiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIHR5cGU9c3RyLCBkZWZhdWx0"
    "PSJhdXRvIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1sb2ctaW50ZXJ2YWwiLCB0eXBlPWludCwgZGVmYXVsdD0yNSkKICAgIGFw"
    "LmFkZF9hcmd1bWVudCgiLS1wYXRpZW5jZSIsIHR5cGU9aW50LCBkZWZhdWx0PTMsCiAgICAgICAgICAgICAgICAgICAgaGVscD0i"
    "VmFsIGl5aWxlc21lenNlIGthYyBldmFsIHNvbnJhIGR1cnVsdXIgKDAgPSBkZXZhbSBldCkiKQogICAgYXAuYWRkX2FyZ3VtZW50"
    "KCItLWV2YWwtaW50ZXJ2YWwiLCB0eXBlPWludCwgZGVmYXVsdD0yMDApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2F2ZS1pbnRl"
    "cnZhbCIsIHR5cGU9aW50LCBkZWZhdWx0PTQwMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXgtc2FtcGxlcyIsIHR5cGU9aW50"
    "LCBkZWZhdWx0PTAsIGhlbHA9IjAgPSB0w7xtw7wiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVm"
    "YXVsdD0xMzM3KQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIHRvcmNoLm1hbnVhbF9zZWVkKGFyZ3Muc2VlZCkKICAg"
    "IHJuZyA9IHJhbmRvbS5SYW5kb20oYXJncy5zZWVkKQoKICAgIGRldmljZSA9IGFyZ3MuZGV2aWNlIGlmIGFyZ3MuZGV2aWNlICE9"
    "ICJhdXRvIiBlbHNlICgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgbG9nZ2VyLmlu"
    "Zm8oIkNpaGF6OiAlcyIsIGRldmljZSkKCiAgICBvdXRfZGlyID0gUGF0aChhcmdzLm91dF9kaXIpCiAgICBvdXRfZGlyLm1rZGly"
    "KHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICAjIFRva2VuaXplcjogbWV2Y3V0IHZvY2FiJ8SxIGt1bGxhbiAoeWVu"
    "aWRlbiBlxJ9pdG1lKQogICAgdm9jYWJfcGF0aCA9IFJPT1QgLyAiY2hlY2twb2ludHMiIC8gInZvY2FiLmpzb24iCiAgICBsb2dn"
    "ZXIuaW5mbygiVm9jYWIgecO8a2xlbml5b3I6ICVzIiwgdm9jYWJfcGF0aCkKICAgIHRvayA9IEJQRVRva2VuaXplci5sb2FkKHZv"
    "Y2FiX3BhdGgpCiAgICB0b2tfcGFkX2lkX2hvbGRlclswXSA9IHRvay5wYWRfaWQKICAgIGxvZ2dlci5pbmZvKCJWb2NhYjogJWQg"
    "dG9rZW4sIGVvcz0lZCBib3M9JWQiLCB0b2sudm9jYWJfc2l6ZSwgdG9rLmVvc19pZCwgdG9rLmJvc19pZCkKCiAgICAjIFZlcmkK"
    "ICAgIHJvd3MgPSBsb2FkX2pzb25sKFBhdGgoYXJncy5kYXRhKSkKICAgIGlmIGFyZ3MubWF4X3NhbXBsZXM6CiAgICAgICAgcm93"
    "cyA9IHJvd3NbOmFyZ3MubWF4X3NhbXBsZXNdCiAgICBsb2dnZXIuaW5mbygiSGFtIMO2cm5lazogJWQiLCBsZW4ocm93cykpCgog"
    "ICAgcmVzdW1lX3BhdGggPSBQYXRoKGFyZ3MucmVzdW1lKQogICAgaWYgbm90IHJlc3VtZV9wYXRoLmV4aXN0cygpOgogICAgICAg"
    "IGxvZ2dlci5lcnJvcigiQ2hlY2twb2ludCBidWx1bmFtYWTEsTogJXMiLCByZXN1bWVfcGF0aCkKICAgICAgICByZXR1cm4KICAg"
    "IGNrcHQgPSB0b3JjaC5sb2FkKHJlc3VtZV9wYXRoLCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdodHNfb25seT1GYWxzZSkKICAg"
    "IHN0YXRlX2tleSA9ICJtb2RlbF9zdGF0ZSIgaWYgIm1vZGVsX3N0YXRlIiBpbiBja3B0IGVsc2UgIm1vZGVsIgogICAgY2ZnID0g"
    "TW9kZWxDb25maWcoKipja3B0WyJjb25maWciXS5fX2RpY3RfXykgaWYgaXNpbnN0YW5jZShja3B0WyJjb25maWciXSwgTW9kZWxD"
    "b25maWcpIFwKICAgICAgICBlbHNlIE1vZGVsQ29uZmlnKCoqY2twdFsiY29uZmlnIl0pCiAgICBibG9ja19zaXplID0gY2ZnLmJs"
    "b2NrX3NpemUKICAgIGxvZ2dlci5pbmZvKCJNb2RlbCBjb25maWc6IHZvY2FiPSVkIGVtYmQ9JWQgbGF5ZXI9JWQgYmxvY2s9JWQi"
    "LAogICAgICAgICAgICAgICAgY2ZnLnZvY2FiX3NpemUsIGNmZy5uX2VtYmQsIGNmZy5uX2xheWVyLCBibG9ja19zaXplKQoKICAg"
    "ICMgRW5jb2RlCiAgICBzYW1wbGVzID0gW10KICAgIHNraXBwZWQgPSAwCiAgICBmb3IgciBpbiByb3dzOgogICAgICAgIGVuYyA9"
    "IGVuY29kZV9leGFtcGxlKHRvaywgclsiaW5zdHJ1Y3Rpb24iXSwgclsiaW5wdXQiXSwgclsib3V0cHV0Il0sIGJsb2NrX3NpemUp"
    "CiAgICAgICAgaWYgZW5jIGlzIE5vbmU6CiAgICAgICAgICAgIHNraXBwZWQgKz0gMQogICAgICAgICAgICBjb250aW51ZQogICAg"
    "ICAgIHNhbXBsZXMuYXBwZW5kKGVuYykKICAgIGxvZ2dlci5pbmZvKCJLdWxsYW7EsWxhYmlsaXI6ICVkIChibG9jaz0lZCdhIHPE"
    "scSfbWF5YW46ICVkKSIsIGxlbihzYW1wbGVzKSwgYmxvY2tfc2l6ZSwgc2tpcHBlZCkKCiAgICBpZiBub3Qgc2FtcGxlczoKICAg"
    "ICAgICBsb2dnZXIuZXJyb3IoIkt1bGxhbsSxbGFiaWxpciDDtnJuZWsgeW9rISIpCiAgICAgICAgcmV0dXJuCgogICAgIyBNb2Rl"
    "bAogICAgbW9kZWwgPSBDb2ZldVRyYW5zZm9ybWVyKGNmZykudG8oZGV2aWNlKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGNr"
    "cHRbc3RhdGVfa2V5XSkKICAgIG1vZGVsLnRyYWluKCkKICAgIG5fcGFyYW1zID0gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2Rl"
    "bC5wYXJhbWV0ZXJzKCkpCiAgICBsb2dnZXIuaW5mbygiTW9kZWwgecO8a2xlbmRpOiAlLjJmTSBwYXJhbWV0cmUiLCBuX3BhcmFt"
    "cyAvIDFlNikKCiAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWFyZ3MubHIsIHdlaWdo"
    "dF9kZWNheT0wLjAxLCBiZXRhcz0oMC45LCAwLjk1KSkKCiAgICAjIFNwbGl0OiB2ZXJpIHPEsXJhbMSxIG9sYWJpbGlyIChJbnN0"
    "cnVjVHVyY2Ega29udS9rYXluYWsgYmF6bMSxIHPEsXJhbMSxIGdlbGl5b3IpLAogICAgIyBidSB5w7x6ZGVuIHZhbGlkYXRpb24g"
    "acOnaW4gInNvbiAlNSIgeWVyaW5lIHNlZWQnbGkga2FyxLHFn3TEsXJtYSBrdWxsYW7EsXlvcnV6LgogICAgc2h1ZmZsZWQgPSBs"
    "aXN0KHNhbXBsZXMpCiAgICByYW5kb20uUmFuZG9tKGFyZ3Muc2VlZCArIDEpLnNodWZmbGUoc2h1ZmZsZWQpCgogICAgbl92YWwg"
    "PSBtYXgobGVuKHNodWZmbGVkKSAvLyAyMCwgMSkKICAgIHZhbF9zYW1wbGVzID0gc2h1ZmZsZWRbOm5fdmFsXQogICAgdHJhaW5f"
    "c2FtcGxlcyA9IHNodWZmbGVkW25fdmFsOl0KICAgIGxvZ2dlci5pbmZvKCJTRlQgdHJhaW46ICVkLCB2YWw6ICVkIiwgbGVuKHRy"
    "YWluX3NhbXBsZXMpLCBsZW4odmFsX3NhbXBsZXMpKQoKICAgICMgQWTEsW0gc2F5xLFzxLEgVFJBSU4ga8O8bWVzaW5kZW4gaGVz"
    "YXBsYW7EsXI7IGFrc2kgaGFsZGUgbG9vcCBiaXIgZXBvY2gndQogICAgIyBhxZ/EsXlvciB2ZSBjb3NpbmUgTFIgZ2Vyw6dlayBl"
    "cG9jaCBzb251bmRhIHPEsWbEsXJhIGlubWVtacWfIG9sdXlvci4KICAgIHN0ZXBzX3Blcl9lcG9jaCA9IG1hdGguY2VpbChsZW4o"
    "dHJhaW5fc2FtcGxlcykgLyBhcmdzLmJhdGNoX3NpemUpCiAgICB0b3RhbF9zdGVwcyA9IHN0ZXBzX3Blcl9lcG9jaCAqIGFyZ3Mu"
    "ZXBvY2hzCiAgICBpZiBhcmdzLm1heF9pdGVyczoKICAgICAgICB0b3RhbF9zdGVwcyA9IG1pbih0b3RhbF9zdGVwcywgYXJncy5t"
    "YXhfaXRlcnMpCiAgICB3YXJtdXAgPSBtaW4oMTAwLCB0b3RhbF9zdGVwcyAvLyAxMCkKCiAgICBkZWYgZ2V0X2xyKGl0OiBpbnQp"
    "IC0+IGZsb2F0OgogICAgICAgIGlmIGl0IDwgd2FybXVwOgogICAgICAgICAgICByZXR1cm4gYXJncy5sciAqIChpdCArIDEpIC8g"
    "bWF4KHdhcm11cCwgMSkKICAgICAgICBwcm9nID0gKGl0IC0gd2FybXVwKSAvIG1heCh0b3RhbF9zdGVwcyAtIHdhcm11cCwgMSkK"
    "ICAgICAgICByZXR1cm4gYXJncy5sciAqIDAuNSAqICgxICsgbWF0aC5jb3MobWF0aC5waSAqIG1pbihwcm9nLCAxLjApKSkKCiAg"
    "ICBsb2dnZXIuaW5mbygiRcSfaXRpbSBiYcWfbMSxeW9yOiAlZCBhZMSxbSAoJWQgZXBvY2ggeCAlZCBhZMSxbSkiLCB0b3RhbF9z"
    "dGVwcywgYXJncy5lcG9jaHMsIHN0ZXBzX3Blcl9lcG9jaCkKICAgIHN0ZXAgPSAwCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBi"
    "ZXN0X3ZhbCA9IGZsb2F0KCJpbmYiKQogICAgZXZhbHNfc2luY2VfaW1wcm92ZSA9IDAKICAgIHN0b3BwZWRfZWFybHkgPSBGYWxz"
    "ZQoKICAgIGZvciBlcG9jaCBpbiByYW5nZShhcmdzLmVwb2Nocyk6CiAgICAgICAgZm9yIHgsIHkgaW4gbWFrZV9iYXRjaGVzKHRy"
    "YWluX3NhbXBsZXMsIGFyZ3MuYmF0Y2hfc2l6ZSwgYmxvY2tfc2l6ZSwgcm5nKToKICAgICAgICAgICAgaWYgc3RlcCA+PSB0b3Rh"
    "bF9zdGVwcyBvciBzdG9wcGVkX2Vhcmx5OgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZm9yIGcgaW4gb3B0LnBh"
    "cmFtX2dyb3VwczoKICAgICAgICAgICAgICAgIGdbImxyIl0gPSBnZXRfbHIoc3RlcCkKICAgICAgICAgICAgeCwgeSA9IHgudG8o"
    "ZGV2aWNlKSwgeS50byhkZXZpY2UpCiAgICAgICAgICAgIF8sIGxvc3MgPSBtb2RlbCh4LCB0YXJnZXRzPXkpCiAgICAgICAgICAg"
    "IGxvc3MuYmFja3dhcmQoKQogICAgICAgICAgICB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8obW9kZWwucGFyYW1ldGVy"
    "cygpLCBhcmdzLmdyYWRfY2xpcCkKICAgICAgICAgICAgb3B0LnN0ZXAoKQogICAgICAgICAgICBvcHQuemVyb19ncmFkKHNldF90"
    "b19ub25lPVRydWUpCiAgICAgICAgICAgIHN0ZXAgKz0gMQoKICAgICAgICAgICAgaWYgc3RlcCAlIGFyZ3MubG9nX2ludGVydmFs"
    "ID09IDA6CiAgICAgICAgICAgICAgICBlbGFwc2VkID0gdGltZS50aW1lKCkgLSB0MAogICAgICAgICAgICAgICAgcHBsID0gbWF0"
    "aC5leHAobWluKGxvc3MuaXRlbSgpLCAyMCkpCiAgICAgICAgICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgICAgICAg"
    "ICAic3RlcCAlNWQvJWQgfCBsb3NzICUuNGYgfCBwcGwgJTcuMmYgfCBsciAlLjJlIHwgJS4yZiB0b2svcyIsCiAgICAgICAgICAg"
    "ICAgICAgICAgc3RlcCwgdG90YWxfc3RlcHMsIGxvc3MuaXRlbSgpLCBwcGwsIGdldF9scihzdGVwKSwKICAgICAgICAgICAgICAg"
    "ICAgICBzdGVwICogeC5udW1lbCgpIC8gZWxhcHNlZCwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgc3RlcCAlIGFy"
    "Z3MuZXZhbF9pbnRlcnZhbCA9PSAwOgogICAgICAgICAgICAgICAgdmwgPSBldmFsdWF0ZShtb2RlbCwgdmFsX3NhbXBsZXMsIGFy"
    "Z3MuYmF0Y2hfc2l6ZSwgYmxvY2tfc2l6ZSwgZGV2aWNlKQogICAgICAgICAgICAgICAgbG9nZ2VyLmluZm8oIiAgPj4gVkFMIGxv"
    "c3MgJS40ZiAocHBsICUuMmYpIiwgdmwsIG1hdGguZXhwKG1pbih2bCwgMjApKSkKICAgICAgICAgICAgICAgIGlmIHZsIDwgYmVz"
    "dF92YWw6CiAgICAgICAgICAgICAgICAgICAgYmVzdF92YWwgPSB2bAogICAgICAgICAgICAgICAgICAgIGV2YWxzX3NpbmNlX2lt"
    "cHJvdmUgPSAwCiAgICAgICAgICAgICAgICAgICAgdG9yY2guc2F2ZSh7CiAgICAgICAgICAgICAgICAgICAgICAgICJtb2RlbF9z"
    "dGF0ZSI6IG1vZGVsLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgICAgICAgICAgICAgImNvbmZpZyI6IGNmZywKICAgICAgICAg"
    "ICAgICAgICAgICAgICAgIml0ZXJhdGlvbiI6IHN0ZXAsCiAgICAgICAgICAgICAgICAgICAgICAgICJ2YWxfbG9zcyI6IHZsLAog"
    "ICAgICAgICAgICAgICAgICAgICAgICAic2Z0IjogVHJ1ZSwKICAgICAgICAgICAgICAgICAgICB9LCBvdXRfZGlyIC8gImNvZmV1"
    "X3NmdC5wdCIpCiAgICAgICAgICAgICAgICAgICAgbG9nZ2VyLmluZm8oIiAgPj4ga2F5ZGVkaWxkaTogY29mZXVfc2Z0LnB0IikK"
    "ICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgIyBWYWwgYXJ0dMSxIC0+IGV6YmVybGVtZSBiYcWfbGFk"
    "xLEuIEVuIGl5aSBtb2RlbCBlbGltaXpkZSwKICAgICAgICAgICAgICAgICAgICAjIGRhaGEgZmF6bGEgZcSfaXRpbSBzYWRlY2Ug"
    "Ym96YXIuCiAgICAgICAgICAgICAgICAgICAgZXZhbHNfc2luY2VfaW1wcm92ZSArPSAxCiAgICAgICAgICAgICAgICAgICAgbG9n"
    "Z2VyLmluZm8oIiAgPj4gdmFsIGFydHTEsSAoJWQvJWQpLCBiZXN0OiAlLjRmIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "ICAgICBldmFsc19zaW5jZV9pbXByb3ZlLCBhcmdzLnBhdGllbmNlLCBiZXN0X3ZhbCkKICAgICAgICAgICAgICAgICAgICBpZiBh"
    "cmdzLnBhdGllbmNlIGFuZCBldmFsc19zaW5jZV9pbXByb3ZlID49IGFyZ3MucGF0aWVuY2U6CiAgICAgICAgICAgICAgICAgICAg"
    "ICAgIGxvZ2dlci5pbmZvKCJFUktFTiBEVVJEVVJNQTogdmFsICVkIGtleiBpeWlsZXNtZWRpLiAiCiAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAgICJFbiBpeWkgbW9kZWwga2F5ZGVkaWxkaS4iLCBhcmdzLnBhdGllbmNlKQogICAgICAgICAgICAg"
    "ICAgICAgICAgICBzdG9wcGVkX2Vhcmx5ID0gVHJ1ZQogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBp"
    "ZiBzdGVwICUgYXJncy5zYXZlX2ludGVydmFsID09IDA6CiAgICAgICAgICAgICAgICB0b3JjaC5zYXZlKHsKICAgICAgICAgICAg"
    "ICAgICAgICAibW9kZWxfc3RhdGUiOiBtb2RlbC5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAgICAgImNvbmZpZyI6IGNm"
    "ZywKICAgICAgICAgICAgICAgICAgICAiaXRlcmF0aW9uIjogc3RlcCwKICAgICAgICAgICAgICAgICAgICAic2Z0IjogVHJ1ZSwK"
    "ICAgICAgICAgICAgICAgIH0sIG91dF9kaXIgLyAiY29mZXVfc2Z0X2xhdGVzdC5wdCIpCiAgICAgICAgaWYgc3RlcCA+PSB0b3Rh"
    "bF9zdGVwcyBvciBzdG9wcGVkX2Vhcmx5OgogICAgICAgICAgICBicmVhawoKICAgIHZsID0gZXZhbHVhdGUobW9kZWwsIHZhbF9z"
    "YW1wbGVzLCBhcmdzLmJhdGNoX3NpemUsIGJsb2NrX3NpemUsIGRldmljZSkKICAgIGxvZ2dlci5pbmZvKCJTT04gdmFsIGxvc3Mg"
    "JS40ZiAocHBsICUuMmYpIiwgdmwsIG1hdGguZXhwKG1pbih2bCwgMjApKSkKCiAgICAjIGNvZmV1X3NmdC5wdCBkYWltYSAiZW4g"
    "aXlpIiBtb2RlbGkgdHV0bWFsxLEuIFNvbiBhZMSxbSBkYWhhIGvDtnTDvHlzZQogICAgIyAoZWFybHkgc3RvcHBpbmcgdmV5YSB2"
    "YWwgYXJ0xLHFn8SxKSDDvHplcmluZSB5YXptYWsgZW4gaXlpIG1vZGVsaSBib3phci4KICAgIGlmIHZsIDwgYmVzdF92YWw6CiAg"
    "ICAgICAgYmVzdF92YWwgPSB2bAogICAgICAgIHRvcmNoLnNhdmUoeyJtb2RlbF9zdGF0ZSI6IG1vZGVsLnN0YXRlX2RpY3QoKSwg"
    "ImNvbmZpZyI6IGNmZywgIml0ZXJhdGlvbiI6IHN0ZXAsCiAgICAgICAgICAgICAgICAgICAgInZhbF9sb3NzIjogdmwsICJzZnQi"
    "OiBUcnVlfSwgb3V0X2RpciAvICJjb2ZldV9zZnQucHQiKQogICAgICAgIGxvZ2dlci5pbmZvKCJTb24gbW9kZWwgZW4gaXlpc2kg"
    "LT4gY29mZXVfc2Z0LnB0IikKICAgIGVsc2U6CiAgICAgICAgbG9nZ2VyLmluZm8oIlNvbiBtb2RlbCBkYWhhIGvDtnTDvCAoJS40"
    "ZiA+PSBiZXN0ICUuNGYpIC0+IGVuIGl5aSBjaGVja3BvaW50IGtvcnVudXlvciIsCiAgICAgICAgICAgICAgICAgICAgdmwsIGJl"
    "c3RfdmFsKQoKICAgIGxvZ2dlci5pbmZvKCJCaXR0aSAtPiAlcyIsIG91dF9kaXIgLyAiY29mZXVfc2Z0LnB0IikKCgppZiBfX25h"
    "bWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="
)
_raw = base64.b64decode(_b64)
_p = pathlib.Path('/content/cofeuai/python/sft.py')
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_raw)
print("gomuldu: python/sft.py -> " + str(_p) + " (" + str(len(_raw)) + " byte)")

In [ ]:
import base64, pathlib
# GOMULU KOPYA: python/prepare_sft_data.py -> /content/cofeuai/python/prepare_sft_data.py
# Dosya GitHub'a push edilmedi; notebook kendi kopyasini yaziyor.
# Ureten: data/sft/build_chat_notebook.py (python/prepare_sft_data.py okunarak)
_b64 = (
    "IiIiQ29mZXUgU0ZUIHZlcmkgaGF6aXJsaWdpOiBTaGFyZUdQVCBUdXJrY2UgKyB0YXNjaWIvdHVya2lzaC1pbnN0cnVjdGlvbi4K"
    "CklraSBrYXluYWdpIGJpcmxlc3RpcmlyOgogIDEuIEFoaXNrYUFJL3NoYXJlZ3B0LXR1cmtpc2ggIC0+IGdlcmNlayBUdXJrY2Ug"
    "c29oYmV0IChjb2tsdSB0dXIpCiAgMi4gdGFzY2liL3R1cmtpc2gtaW5zdHJ1Y3Rpb24gLT4ga2lzYSBzb3J1L2NldmFwIChpbnN0"
    "cnVjdGlvbi9pbnB1dC9vdXRwdXQpCgpTb2hiZXQgdmVyaXNpIGNvZ3UgemFtYW4gPjUxMiB0b2tlbiwgbyB5dXpkZW4gaGVyIGtv"
    "bnVzbWEgdGVrIHR1cmx1CihpbnN0cnVjdGlvbiwgb3V0cHV0KSBjaWZ0bGVyaW5lIGR1emxlc3RpcmlsaXIuIEJveWxlY2UgZWdp"
    "dGltIHNpcmFzaW5kYQpzYXJ0bmFtZSB0YW0gb2xhcmFrIG5lIG9sYWNhaz8gQ3Vua3UgaW5mZXJlbmNlJ2RhIGRhIHRlayB0dXIg"
    "c29ydXlvcnV6LgoKQ2lrdGkgc2VtYXNpIHNmdC5weSBpbGUgYXluaTogaW5zdHJ1Y3Rpb24gLyBpbnB1dCAvIG91dHB1dC4KCkt1"
    "bGxhbmltOgogIHB5dGhvbiBweXRob24vcHJlcGFyZV9zZnRfZGF0YS5weSAtLW91dCAvY29udGVudC9zZnRfZGF0YS5qc29ubAoi"
    "IiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBv"
    "cwppbXBvcnQgcmFuZG9tCmltcG9ydCByZQppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB1cmxsaWIucmVxdWVz"
    "dApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEl0ZXJhYmxlLCBJdGVyYXRvciwgTGlzdCwgT3B0"
    "aW9uYWwsIFR1cGxlCgpTT1VSQ0VTID0gewogICAgInNoYXJlZ3B0IjogImh0dHBzOi8vaHVnZ2luZ2ZhY2UuY28vZGF0YXNldHMv"
    "QWhpc2thQUkvc2hhcmVncHQtdHVya2lzaC9yZXNvbHZlL21haW4vZGF0YXNldC5qc29uIiwKICAgICJ0YXNjaWIiOiAiaHR0cHM6"
    "Ly9odWdnaW5nZmFjZS5jby9kYXRhc2V0cy90YXNjaWIvdHVya2lzaC1pbnN0cnVjdGlvbi9yZXNvbHZlL21haW4vZGF0YS90dXJr"
    "aXNoX2luc3RydWN0aW9uX2RhdGFzZXQuanNvbmwiLAp9CgpNSU5fQ0hBUlMgPSAyCk1BWF9PVVRfQ0hBUlMgPSAxMjAwICAgICAg"
    "IyA1MTIgdG9rZW4nYSBzaWdtYXNpbmEgeWFyZGltY2kgb2x1cgpNQVhfSU5TVFJfQ0hBUlMgPSA4MDAKCgpkZWYgbG9nKG1zZzog"
    "c3RyKSAtPiBOb25lOgogICAgcHJpbnQobXNnLCBmbHVzaD1UcnVlKQoKCmRlZiBkb3dubG9hZCh1cmw6IHN0ciwgZGVzdDogUGF0"
    "aCwgdGltZW91dDogaW50ID0gMTgwMCkgLT4gUGF0aDoKICAgICIiIkRvc3lheWkgaW5kaXJpcjsgdmFyc2EgdGVrcmFyIGluZGly"
    "bWV6LiIiIgogICAgZGVzdC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgaWYgZGVzdC5leGlz"
    "dHMoKSBhbmQgZGVzdC5zdGF0KCkuc3Rfc2l6ZSA+IDA6CiAgICAgICAgbG9nKGYiICB6YXRlbiB2YXI6IHtkZXN0Lm5hbWV9ICh7"
    "ZGVzdC5zdGF0KCkuc3Rfc2l6ZS8xZTY6LjFmfSBNQikiKQogICAgICAgIHJldHVybiBkZXN0CiAgICBsb2coZiIgIGluZGlyaWxp"
    "eW9yOiB7dXJsfSIpCiAgICB0bXAgPSBkZXN0LndpdGhfc3VmZml4KGRlc3Quc3VmZml4ICsgIi5wYXJ0IikKICAgICMgY3VybCBp"
    "bGU6IEhGIHlvbmxlbmRpcm1lbGVyaW5pIGl5aSB0YWtpcCBlZGl5b3IgdmUgaWxlcmxlbWUgZ29zdGVyaXlvcgogICAgcmMgPSBz"
    "dWJwcm9jZXNzLnJ1bigKICAgICAgICBbImN1cmwiLCAiLXNMIiwgIi0tZmFpbCIsICItLW1heC10aW1lIiwgc3RyKHRpbWVvdXQp"
    "LCAiLW8iLCBzdHIodG1wKSwgdXJsXQogICAgKS5yZXR1cm5jb2RlCiAgICBpZiByYyAhPSAwIG9yIG5vdCB0bXAuZXhpc3RzKCkg"
    "b3IgdG1wLnN0YXQoKS5zdF9zaXplID09IDA6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiaW5kaXJtZSBiYXNhcmlzaXog"
    "KHt1cmx9KSwgY3VybCBleGl0PXtyY30iKQogICAgdG1wLnJlbmFtZShkZXN0KQogICAgbG9nKGYiICBpbmRpcmlsZGk6IHtkZXN0"
    "Lm5hbWV9ICh7ZGVzdC5zdGF0KCkuc3Rfc2l6ZS8xZTY6LjFmfSBNQikiKQogICAgcmV0dXJuIGRlc3QKCgojIC0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gdGVtaXpsZW1lCgpfV1MgPSByZS5j"
    "b21waWxlKHIiXHMrIikKCgpkZWYgaGFzX2RlZ2VuZXJhdGVfcmVwZWF0KHRleHQ6IHN0ciwgbWF4X3BlcmlvZDogaW50ID0gOCwg"
    "dGltZXM6IGludCA9IDMpIC0+IGJvb2w6CiAgICAiIiJLaXNhIGJpciBrZWxpbWUgYmxvZ3VudW4gYXJrYSBhcmtheWEgdGVrcmFy"
    "bGFuaXAgdGVrcmFybGFtYWRpZ2kuCgogICAgVXJldGljaSB2ZXJpZGUgInTEsWtwxLEgYmlyIHTEsWtwxLEgYmlyIHTEsWtwxLEi"
    "IGdpYmkgYm96dWtsdWtsYXIgdmFyOyBtb2RlbGluCiAgICBkZSB1cmV0dGlnaSB0aXAgYmlyIHRla3JhciBkb25ndXN1bnUgYmVz"
    "bGVtZW1layBpY2luIGVsaXlvcnV6LgoKICAgIFRlayBiaXIgc2FiaXQgYmxvZyBib3l1dHUgKG9ybi4gMykgeWV0bWV6OiBib3p1"
    "a2x1a2xhciAyIGtlbGltZWxpawogICAgKCJiaXIgdMSxa3DEsSBiaXIgdMSxa3DEsSBiaXIgdMSxa3DEsSIpIHlhIGRhIDUga2Vs"
    "aW1lbGlrIHBlcml5b3RsYXJkYSBkYQogICAgZ29ydWx1eW9yLiBCdSB5dXpkZW4gMi4ubWF4X3BlcmlvZCBhcmFzaSB0dW0gcGVy"
    "aXlvdGxhcmkgdGFyYXJpei4KICAgICIiIgogICAgd29yZHMgPSB0ZXh0LnNwbGl0KCkKICAgIG4gPSBsZW4od29yZHMpCiAgICBm"
    "b3IgcGVyaW9kIGluIHJhbmdlKDIsIG1heF9wZXJpb2QgKyAxKToKICAgICAgICBuZWVkID0gcGVyaW9kICogdGltZXMKICAgICAg"
    "ICBpZiBuIDwgbmVlZDoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBmb3IgaSBpbiByYW5nZShuIC0gbmVlZCArIDEpOgogICAg"
    "ICAgICAgICBibG9jayA9IHdvcmRzW2k6aSArIHBlcmlvZF0KICAgICAgICAgICAgaWYgYWxsKHdvcmRzW2kgKyBqICogcGVyaW9k"
    "OmkgKyAoaiArIDEpICogcGVyaW9kXSA9PSBibG9jawogICAgICAgICAgICAgICAgICAgZm9yIGogaW4gcmFuZ2UoMSwgdGltZXMp"
    "KToKICAgICAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICByZXR1cm4gRmFsc2UKCgpkZWYgY2xlYW4odGV4dDogT3B0aW9uYWxb"
    "c3RyXSkgLT4gc3RyOgogICAgaWYgbm90IHRleHQ6CiAgICAgICAgcmV0dXJuICIiCiAgICAjIFUrMjAwRCB6ZXJvLXdpZHRoIGpv"
    "aW5lciB1cmV0aWNpIG1ldGlubGVyZGUgYXJ0aWsga2FsaXlvcgogICAgdGV4dCA9IHRleHQucmVwbGFjZSgiXHUyMDBkIiwgIiIp"
    "LnN0cmlwKCkKICAgIHJldHVybiBfV1Muc3ViKCIgIiwgdGV4dCkKCgpkZWYgdXNhYmxlKGluc3RyOiBzdHIsIG91dDogc3RyKSAt"
    "PiBib29sOgogICAgaWYgbGVuKGluc3RyKSA8IE1JTl9DSEFSUyBvciBsZW4ob3V0KSA8IE1JTl9DSEFSUzoKICAgICAgICByZXR1"
    "cm4gRmFsc2UKICAgIGlmIGxlbihpbnN0cikgPiBNQVhfSU5TVFJfQ0hBUlMgb3IgbGVuKG91dCkgPiBNQVhfT1VUX0NIQVJTOgog"
    "ICAgICAgIHJldHVybiBGYWxzZQogICAgaWYgaGFzX2RlZ2VuZXJhdGVfcmVwZWF0KG91dCk6CiAgICAgICAgcmV0dXJuIEZhbHNl"
    "CiAgICByZXR1cm4gVHJ1ZQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLSBrYXluYWtsYXIKCmRlZiBpdGVyX3NoYXJlZ3B0KHBhdGg6IFBhdGgpIC0+IEl0ZXJhdG9yW1R1cGxlW3N0ciwg"
    "c3RyXV06CiAgICAiIiJIZXIga29udXNtYWRhbiAoaHVtYW4gdHVydSwgZ3B0IGNldmFiaSkgY2lmdGxlcmkgdXJldGlyLgoKICAg"
    "IEtvbnVzbWEgc2lyYXNpeWxhIHNpc3RlbSArIGh1bWFuL2dwdCAuLi4gc2VrbGluZGUuIEJpciB0dXIgY2lmdGksIGJpcgogICAg"
    "b25jZWtpIGh1bWFuIGNldmFiaW5pIHRha2lwIGVkZW4gaWxrIGdwdCBtZXNhamkgb2x1ci4KICAgICIiIgogICAgd2l0aCBvcGVu"
    "KHBhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgY29udnMgPSBqc29uLmxvYWQoZikKICAgIGZvciBjb252IGlu"
    "IGNvbnZzOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGNvbnYsIGxpc3QpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAg"
    "IHBlbmRpbmc6IE9wdGlvbmFsW3N0cl0gPSBOb25lCiAgICAgICAgZm9yIG1zZyBpbiBjb252OgogICAgICAgICAgICBpZiBub3Qg"
    "aXNpbnN0YW5jZShtc2csIGRpY3QpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcm9sZSwgdmFsID0gbXNn"
    "LmdldCgiZnJvbSIpLCBjbGVhbihtc2cuZ2V0KCJ2YWx1ZSIpKQogICAgICAgICAgICBpZiByb2xlID09ICJodW1hbiIgYW5kIHZh"
    "bDoKICAgICAgICAgICAgICAgIHBlbmRpbmcgPSB2YWwgICAgICAgICAgICAgICAgICAgICAgIyBzb3J1eXUgYmlyYWt0aXIsIGNl"
    "dmFiaSBiZWtsZQogICAgICAgICAgICBlbGlmIHJvbGUgPT0gImdwdCIgYW5kIHZhbCBhbmQgcGVuZGluZzoKICAgICAgICAgICAg"
    "ICAgIHlpZWxkIHBlbmRpbmcsIHZhbAogICAgICAgICAgICAgICAgcGVuZGluZyA9IE5vbmUKCgpkZWYgaXRlcl90YXNjaWIocGF0"
    "aDogUGF0aCkgLT4gSXRlcmF0b3JbVHVwbGVbc3RyLCBzdHIsIHN0cl1dOgogICAgIiIiKGluc3RydWN0aW9uLCBpbnB1dCwgb3V0"
    "cHV0KSB1Y2x1c3UgdXJldGlyLiIiIgogICAgd2l0aCBvcGVuKHBhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAg"
    "Zm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAgICAgICBpZiBub3QgbGluZToKICAg"
    "ICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG9iaiA9IGpzb24ubG9hZHMobGlu"
    "ZSkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGluc3Ry"
    "ID0gY2xlYW4ob2JqLmdldCgiaW5zdHJ1Y3Rpb24iKSkKICAgICAgICAgICAgZXh0cmEgPSBjbGVhbihvYmouZ2V0KCJpbnB1dCIp"
    "KQogICAgICAgICAgICBvdXQgPSBjbGVhbihvYmouZ2V0KCJvdXRwdXQiKSkKICAgICAgICAgICAgaWYgaW5zdHIgYW5kIG91dDoK"
    "ICAgICAgICAgICAgICAgIHlpZWxkIGluc3RyLCBleHRyYSwgb3V0CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gdG9wbGFtYQoKZGVmIHdyaXRlX2pzb25sKHBhdGg6IFBhdGgsIHJv"
    "d3M6IEl0ZXJhYmxlW2RpY3RdKSAtPiBpbnQ6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFy"
    "ZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbiA9IDAKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYt"
    "OCIpIGFzIGY6CiAgICAgICAgZm9yIHIgaW4gcm93czoKICAgICAgICAgICAgZi53cml0ZShqc29uLmR1bXBzKHIsIGVuc3VyZV9h"
    "c2NpaT1GYWxzZSkgKyAiXG4iKQogICAgICAgICAgICBuICs9IDEKICAgIHJldHVybiBuCgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAg"
    "ICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCByZXF1aXJlZD1UcnVl"
    "LCBoZWxwPSJjaWt0aSBqc29ubCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2FjaGUtZGlyIiwgZGVmYXVsdD0iL2NvbnRlbnQv"
    "c2Z0X3JhdyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbWF4LXNoYXJlZ3B0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjAwMDApCiAg"
    "ICBhcC5hZGRfYXJndW1lbnQoIi0tbWF4LXRhc2NpYiIsIHR5cGU9aW50LCBkZWZhdWx0PTYwMDAwKQogICAgYXAuYWRkX2FyZ3Vt"
    "ZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0xMzM3KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXN0YXRzLW9ubHkiLCBh"
    "Y3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InNhZGVjZSBpc3RhdGlzdGlrIHlheiwgZG9zeWEg"
    "dXJldG1lIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBybmcgPSByYW5kb20uUmFuZG9tKGFyZ3Muc2VlZCkKICAg"
    "IGNhY2hlID0gUGF0aChhcmdzLmNhY2hlX2RpcikKCiAgICBsb2coIj0iICogNjApCiAgICBsb2coIjEpIFNoYXJlR1BUIFR1cmtj"
    "ZSAoZ2VyY2VrIHNvaGJldCkiKQogICAgc2dfcGF0aCA9IGRvd25sb2FkKFNPVVJDRVNbInNoYXJlZ3B0Il0sIGNhY2hlIC8gInNo"
    "YXJlZ3B0LXR1cmtpc2guanNvbiIpCiAgICBsb2coIjIpIHRhc2NpYi90dXJraXNoLWluc3RydWN0aW9uIChraXNhIHNvcnUvY2V2"
    "YXApIikKICAgIHRzX3BhdGggPSBkb3dubG9hZChTT1VSQ0VTWyJ0YXNjaWIiXSwgY2FjaGUgLyAidHVya2lzaF9pbnN0cnVjdGlv"
    "bl9kYXRhc2V0Lmpzb25sIikKCiAgICBzZWVuID0gc2V0KCkKICAgIHNoYXJlZ3B0OiBMaXN0W2RpY3RdID0gW10KICAgIHRvdGFs"
    "ID0gZHJvcHBlZCA9IDAKICAgIGZvciBpbnN0ciwgb3V0IGluIGl0ZXJfc2hhcmVncHQoc2dfcGF0aCk6CiAgICAgICAgdG90YWwg"
    "Kz0gMQogICAgICAgIGlmIG5vdCB1c2FibGUoaW5zdHIsIG91dCk6CiAgICAgICAgICAgIGRyb3BwZWQgKz0gMQogICAgICAgICAg"
    "ICBjb250aW51ZQogICAgICAgIGtleSA9IChpbnN0cls6MTIwXSwgb3V0WzoxMjBdKQogICAgICAgIGlmIGtleSBpbiBzZWVuOgog"
    "ICAgICAgICAgICBkcm9wcGVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzZWVuLmFkZChrZXkpCiAgICAgICAg"
    "c2hhcmVncHQuYXBwZW5kKHsiaW5zdHJ1Y3Rpb24iOiBpbnN0ciwgImlucHV0IjogIiIsICJvdXRwdXQiOiBvdXQsCiAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAic291cmNlIjogInNoYXJlZ3B0LXRyIn0pCiAgICBsb2coZiIgIGhhbSBjaWZ0IHt0b3RhbH0gfCBl"
    "bGVuZW4ge2Ryb3BwZWR9IHwga2FsYW4ge2xlbihzaGFyZWdwdCl9IikKCiAgICB0YXNjaWI6IExpc3RbZGljdF0gPSBbXQogICAg"
    "dG90YWwgPSBkcm9wcGVkID0gMAogICAgZm9yIGluc3RyLCBleHRyYSwgb3V0IGluIGl0ZXJfdGFzY2liKHRzX3BhdGgpOgogICAg"
    "ICAgIHRvdGFsICs9IDEKICAgICAgICBpZiBub3QgdXNhYmxlKGluc3RyLCBvdXQpOgogICAgICAgICAgICBkcm9wcGVkICs9IDEK"
    "ICAgICAgICAgICAgY29udGludWUKICAgICAgICBrZXkgPSAoaW5zdHJbOjEyMF0sIG91dFs6MTIwXSkKICAgICAgICBpZiBrZXkg"
    "aW4gc2VlbjoKICAgICAgICAgICAgZHJvcHBlZCArPSAxCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgc2Vlbi5hZGQoa2V5"
    "KQogICAgICAgIHRhc2NpYi5hcHBlbmQoeyJpbnN0cnVjdGlvbiI6IGluc3RyLCAiaW5wdXQiOiBleHRyYSwgIm91dHB1dCI6IG91"
    "dCwKICAgICAgICAgICAgICAgICAgICAgICAic291cmNlIjogInRhc2NpYiJ9KQogICAgbG9nKGYiICBoYW0gc2F0aXIge3RvdGFs"
    "fSB8IGVsZW5lbiB7ZHJvcHBlZH0gfCBrYWxhbiB7bGVuKHRhc2NpYil9IikKCiAgICBybmcuc2h1ZmZsZShzaGFyZWdwdCkKICAg"
    "IHJuZy5zaHVmZmxlKHRhc2NpYikKICAgIGlmIGFyZ3MubWF4X3NoYXJlZ3B0OgogICAgICAgIHNoYXJlZ3B0ID0gc2hhcmVncHRb"
    "OmFyZ3MubWF4X3NoYXJlZ3B0XQogICAgaWYgYXJncy5tYXhfdGFzY2liOgogICAgICAgIHRhc2NpYiA9IHRhc2NpYls6YXJncy5t"
    "YXhfdGFzY2liXQoKICAgIHJvd3MgPSBzaGFyZWdwdCArIHRhc2NpYgogICAgcm5nLnNodWZmbGUocm93cykKCiAgICBsb2coIiIp"
    "CiAgICBsb2coZiJ0b3BsYW06IHtsZW4ocm93cyl9IChzaGFyZWdwdCB7bGVuKHNoYXJlZ3B0KX0gKyB0YXNjaWIge2xlbih0YXNj"
    "aWIpfSkiKQogICAgbG9nKCI9IiAqIDYwKQogICAgaWYgYXJncy5zdGF0c19vbmx5OgogICAgICAgIGZvciByIGluIHJvd3NbOjVd"
    "OgogICAgICAgICAgICBsb2coZiIgIFt7clsnc291cmNlJ119XSB7clsnaW5zdHJ1Y3Rpb24nXVs6NjBdfSAtPiB7clsnb3V0cHV0"
    "J11bOjYwXX0iKQogICAgICAgIHJldHVybiAwCgogICAgbiA9IHdyaXRlX2pzb25sKFBhdGgoYXJncy5vdXQpLCByb3dzKQogICAg"
    "bG9nKGYieWF6aWxkaToge2FyZ3Mub3V0fSAoe259IHNhdGlyKSIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21h"
    "aW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkp"
)
_raw = base64.b64decode(_b64)
_p = pathlib.Path('/content/cofeuai/python/prepare_sft_data.py')
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_bytes(_raw)
print("gomuldu: python/prepare_sft_data.py -> " + str(_p) + " (" + str(len(_raw)) + " byte)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive baglandi")

In [ ]:
import os, shutil, glob, zipfile
from pathlib import Path

REPO = Path('/content/cofeuai')
(REPO / 'checkpoints').mkdir(parents=True, exist_ok=True)
(REPO / 'python').mkdir(parents=True, exist_ok=True)

# train.py modeli uc isimle kaydediyor; hepsini kabul et, tum Drive'i tara.
CANDIDATES = ["cofeu.pt", "cofeu_best.pt", "cofeu_latest.pt"]

def find(names):
    for name in names:
        for pat in (f"/content/drive/MyDrive/{name}",
                    f"/content/drive/MyDrive/cofeuai/checkpoints/{name}",
                    f"/content/drive/MyDrive/cofeuai/{name}",
                    f"/content/drive/MyDrive/checkpoints/{name}"):
            if os.path.exists(pat):
                return Path(pat)
    for name in names:
        hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
        if hits:
            return Path(hits[0])
    return None

src = find(CANDIDATES)
print("=== MODEL ARAMASI ===")
for n in CANDIDATES:
    hits = glob.glob(f"/content/drive/MyDrive/**/{n}", recursive=True)
    print(f"  {n:18s} {'BULUNDU: ' + hits[0] if hits else 'yok'}")
if src is None:
    raise FileNotFoundError(
        "Drive'da model checkpoint'i yok. Yukleyin:\n"
        "  ~/Desktop/CofeuAI/checkpoints/cofeu.pt\n"
        "  -> /content/drive/MyDrive/cofeuai/checkpoints/cofeu.pt")
print(f"secilen: {src}\n")
shutil.copy(src, REPO / 'checkpoints' / 'cofeu.pt')
print(f"cofeu.pt ({os.path.getsize(REPO/'checkpoints'/'cofeu.pt')/1e6:.1f} MB) hazir")

vsrc = find(["vocab.json"])
if vsrc is None:
    raise FileNotFoundError("Drive'da vocab.json yok.")
shutil.copy(vsrc, REPO / 'checkpoints' / 'vocab.json')
print(f"vocab.json <- {vsrc}")

In [ ]:
# Veri hazirligi. Script repo ile birlikte gelir; internet gerekir
# (ShareGPT 31 MB + tascib 390 MB). Onceden /content/sft_raw'a indirilmis
# olurlarsa tekrar indirmez.
#
# Sohbetleri tek turlu (insan, asistan) ciftlerine duzlestirir; medyan 12
# tur oldugu icin sohbetleri oldugu gibi 512 token'a sigmaz.
import subprocess, sys

cmd = [
    sys.executable, "python/prepare_sft_data.py",
    "--out", "/content/sft_data.jsonl",
    "--cache-dir", "/content/sft_raw",
    "--max-sharegpt", "60000",
    "--max-tascib", "60000",
]
print(" ".join(cmd), "\n")
r = subprocess.run(cmd, cwd="/content/cofeuai")
if r.returncode != 0:
    raise RuntimeError(f"veri hazirligi basarisiz, exit={r.returncode}")
print("\nVERI HAZIR")

In [ ]:
# lr 3e-5, 1 epoch. Onceki denemeler 5e-5'te ezberlemeye yol acti.
# Ciktiyi Dogrudan Drive'a yaziyoruz; oturum duserse checkpoint kalir.
# sft.py log DOSYASI yazmiyor, o yuzden ciktiyi hem ekrana hem train.log'a
# yaziyoruz.
import subprocess, sys
from pathlib import Path

# Egitim ayri surecte basliyor; CUDA orada da gecerli olmali.
# Bu olmadan sessizce CPU'ya dustu ve 'not compiled with CUDA'
# hatasini 116K ornek saydiktan sonra verdi.
rc = subprocess.run([sys.executable, "-c",
    "import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)"],
    capture_output=True, text=True)
assert rc.returncode == 0, (
    "Bu ortamda CUDA yok. Onceki GPU kontrol hucrelerini calistir; "
    "sadece bu hucreyi tekrar calistirmak yetmez.")

OUT_DIR = Path('/content/drive/MyDrive/cofeuai/sft_chat')
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG = OUT_DIR / 'train.log'

cmd = [
    sys.executable, "python/sft.py",
    "--data", "/content/sft_data.jsonl",
    "--resume", "/content/cofeuai/checkpoints/cofeu.pt",
    "--out-dir", str(OUT_DIR),
    "--epochs", "1",
    "--batch-size", "16", "--lr", "3e-5", "--patience", "3",
    "--eval-interval", "200", "--save-interval", "500",
    "--log-interval", "50", "--device", "cuda",
]
print("calistiriliyor:", " ".join(cmd), "\n")
print(f"log -> {LOG}\n")

with open(LOG, "w", encoding="utf-8", buffering=1) as logf:
    logf.write(f"$ {' '.join(cmd)}\n\n")
    p = subprocess.Popen(cmd, cwd="/content/cofeuai",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
    rc = p.wait()

if rc != 0:
    raise RuntimeError(f"egitim basarisiz, exit={rc}. Log: {LOG}")
print(f"\nEGITIM TAMAM | log: {LOG}")

In [ ]:
import sys, torch
sys.path.insert(0, '/content/cofeuai/python')
from model import CofeuTransformer, ModelConfig
from tokenizer import BPETokenizer
import sft

CKPT = '/content/drive/MyDrive/cofeuai/sft_chat/cofeu_sft.pt'
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
print("iteration:", ck.get('iteration'), "| val:", ck.get('val_loss'))

cfg = ModelConfig(**ck['config'].__dict__)
m = CofeuTransformer(cfg)
m.load_state_dict(ck['model_state'])
# Kernel'de torch eski surumde kalabilir (CPU-only olabilir); uretim
# testi CPU'da da yavas da olsa calisir, sonucu gormek icin yeterli.
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("uretim cihazi:", DEV)
m.eval().to(DEV)
tok = BPETokenizer.load('/content/cofeuai/checkpoints/vocab.json')

for q in ("Merhaba, nasılsın?",
          "Türkiye'nin başkenti neresidir?",
          "Fotosentez nasıl çalışır?",
          "Bana kısa bir motivasyon cümlesi söyle."):
    p, _ = sft.build_text(q, "", "")
    x = torch.tensor([tok.encode(p)], dtype=torch.long).to(DEV)
    torch.manual_seed(42)
    out = m.generate(x, 120, temperature=0.7, top_k=40, eos_token_id=tok.eos_id)
    print(f"\n>>> {q}\n{tok.decode(out[0, len(x[0]):].tolist())}")

In [ ]:
import subprocess
from pathlib import Path
OUT_DIR = Path('/content/drive/MyDrive/cofeuai/sft_chat')

print(subprocess.run(["bash", "-c", f"tail -25 {OUT_DIR}/train.log"],
                     capture_output=True, text=True).stdout)
print("\n--- dosyalar ---")
print(subprocess.run(["bash", "-c", f"ls -la {OUT_DIR}"],
                     capture_output=True, text=True).stdout)

subprocess.run(["zip", "-j", "/content/cofeu_sft_chat.zip", str(OUT_DIR / 'cofeu_sft.pt')])
print("zip: /content/cofeu_sft_chat.zip  (Colab Files sekmesinden indir)")